# Round 4, step 6 — the vocabulary A/B: old tokens against scheme H

**One variable: the token vocabulary.** The control trains with `--vocab old` (100 ids, every
checkpoint through Round 3). The H arm trains with `--vocab h` (116 ids: 14 common pitches fused into
one token each, plus `'`, `16`, `32`). Everything else is the same for both:

| | both arms |
|---|---|
| synthetic | `strips_v7_final`, **unchanged** — no render this round (owner, 2026-09-03) |
| real | `strips_h1` with the owner's verdicts promoted, at `:4` → real ≈ **30.8%** of stage-2 batches (Run A was 32.9%; `:4` is the repeat closest to ~33%, the rule Run B used) |
| left out of both | the labels over **99 ids under the old vocabulary** (4 at build time — `collate` truncates there, so the control cannot hold them; owner, 2026-09-07). The list travels in the zip |
| stage 2 | **4,000 steps**, lr 1e-5, warmup 100, batch 16 — Run A's recipe |
| checkpoint pick | **free-running corrections** on `_realval_v2r` + `_tupletval` (the `best-edits` tag), decoded to **100** ids — minus the strips whose song is in the synthetic TRAIN split (29 of 295 when measured; owner, 2026-09-16) |

## Stage 1 — the control reuses Round 3's, the H arm runs its own

- **Control:** `r3-final-stage1/best` from Round 3 — the same corpus, split, vocabulary and steps, so a
  re-run would be the same job with a different random draw (owner, 2026-09-16). Saves ~2.5 h.
- **H arm:** must train its own stage 1 from base with `--vocab h` (~2.5 h on an L4), because its
  tokens differ from the very first step.

**GPU total ≈ 2.5 h + 2 × ~1.7 h**, plus the selector's decode at every eval. Two tabs work:
tab 1 runs the control's stage 2; tab 2 runs the H stage 1 and then the H stage 2.

## ⛔ Before you read ANY number: edits are not the same size in the two vocabularies

H spells a note in fewer ids, so **the same mistake costs H about 1.1 fewer edits per strip** when
each model is counted in its own ids (`scripts/rung3/score_vocab_check.py`). That is ~9× real-val's
whole noise band.

- The `best-edits` numbers in the two training logs are each in their **own** ids: use them to
  compare checkpoints **inside one run**, never across the two runs.
- Every A/B number comes from `paired_arm_score.py --score-vocab old`, which re-counts both arms in
  the old ids. The script **refuses** two different vocabularies without it.
- ⚠ `eval_omr.py` (the exam scorer) has **no** shared-scale mode yet — it prints a warning for an
  H checkpoint. Do not read the exam with it on an H model until that is built.

In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK =====
STRIPS = 'data/synthetic/strips_v7_final'   # unchanged — no render this round
ZIP = 'tnc_round4_colab.zip'                # built by scripts/make_round4_colab_zip.sh
DRIVE = '/content/drive/MyDrive/tnc'
REAL = 'data/real/rung3/strips_h1'          # the owner's verdicts promoted; >99-old-id labels left out
REPEAT = 4                                  # -> real ~30.8% of stage-2 batches. See the header.
SELECT = '--select-dir data/real/rung3/_realval_v2r --select-dir data/real/rung3/_tupletval'
SELECT_MAX = 100    # ⚠ NOT the old default 60: 17 of the 295 selection labels need 60+ old ids
STAGE1_CTL = f'{DRIVE}/r3-final-stage1/best'   # Round 3's — reused for the control (owner, 2026-09-16)
STAGE1_H = f'{DRIVE}/r4-h-stage1'              # trained here, --vocab h
print(STRIPS, ZIP, f'| real {REAL}:{REPEAT} | select max {SELECT_MAX}')

In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk, CHECK it, unzip.
# ⛔ The Drive mount drops most often while a fresh upload is still processing; `cp` then leaves a
# TRUNCATED file and the first visible error is three cells later. Fail here with the real reason.
import os, subprocess, json
ZIP_BYTES = 854222642   # written by scripts/make_round4_colab_zip.sh — never edit by hand
assert ZIP_BYTES > 0, 'ZIP_BYTES is 0: build the zip with scripts/make_round4_colab_zip.sh, which writes it here'

src = f'{DRIVE}/{ZIP}'
assert os.path.exists(src), f'{src} not on Drive (upload unfinished, or the mount is dead — restart and re-mount)'
have = os.path.getsize(src)
assert have == ZIP_BYTES, f'Drive has {have:,} bytes, expected {ZIP_BYTES:,}: the upload is incomplete or still syncing'
!rsync --progress {DRIVE}/{ZIP} /content/
got = os.path.getsize(f'/content/{ZIP}')
assert got == ZIP_BYTES, f'copied {got:,} of {ZIP_BYTES:,} bytes — re-run this cell, rsync resumes'
assert subprocess.run(['unzip', '-tq', f'/content/{ZIP}']).returncode == 0, 'the copied zip fails its integrity test'
print('zip integrity OK')

!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# the render is label-free, so render_config.json is the only place it is checkable
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
want = dict(legacyTupletMark=False, thinSharps=True, printNoise=False, staccatoNoise=True,
            concaveTuplet=True, usulBarline=True, maxMeasures=None)
assert {k: cfg.get(k) for k in want} == want, cfg
print('render: Round 3 final flags, unchanged')

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== WHAT IS ON DISK — read these lines before spending a GPU hour =====
%cd /content/tnc
import sys, json
sys.path.insert(0, 'src/vision')
from augment import Augmenter
from data import StripDataset, vocabulary
a = Augmenter(seed=7)
assert (a.photo_share, a.scan_share) == (0.35, 0.0), (a.photo_share, a.scan_share)
print(f'mix: screenshot {1 - a.photo_share - a.scan_share:.2f} / photo {a.photo_share} / scan {a.scan_share}  (default)')
assert (len(vocabulary('old')), len(vocabulary('h'))) == (25, 42), 'vocabulary lists changed — this is not the kit that was measured'

real = StripDataset(REAL).strips
left_out = [l.split('\t')[0] for l in open(f'{REAL}/r4_excluded_over99_old.txt') if not l.startswith('#')]
assert not ({s.image_path.name for s in real} & set(left_out)), 'a left-out label is still in the manifest'
print(f'real pool: {len(real)} strips, {len(left_out)} left out of BOTH arms (> 99 old ids)')
for d in ['data/real/rung3/_realval_v2r', 'data/real/rung3/_tupletval']:
    print(f'selection pool {d}: {len(StripDataset(d).strips)} strips')

In [ ]:
# SHAKEOUT (~6 min): 150 tiny steps from BASE, ONE PER ARM — a wiring smoke, not a result.
# Expect, per arm:
#   `vocab: old (+25 tokens -> 100 ids)`  /  `vocab: h (+42 tokens -> 116 ids)`  (42 listed; `'` already had an id)
#   `real pool data/real/rung3/strips_h1: N train x4 / M val strips`
#   `exam-disjointness OK`
#   `selection: left out N of M strips (K pieces whose song is in the synthetic TRAIN split)` — EXPECTED
#   (owner, 2026-09-16); a REFUSAL here means a selection piece is in the REAL pool, which is a real defect
#   val loss FALLING, and a `best-edits` line at each eval
%cd /content/tnc
for vocab in ['old', 'h']:
    print('=' * 30, 'shakeout', vocab)
    !python -u src/vision/train.py --vocab {vocab} --strips-dir {STRIPS} --split data/split_v4.json \
        --real-dir {REAL}:{REPEAT} {SELECT} --select-max-length {SELECT_MAX} --limit-select 16 \
        --every-share 0.15 --out-dir /content/shake-{vocab} \
        --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
        --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CONTROL, STAGE 1 — ⛔ DO NOT TRAIN. REUSE ROUND 3's. =====
# Same corpus, split, vocabulary and 6,000 steps as Round 3's final run (owner, 2026-09-16).
import os
assert os.path.isdir(STAGE1_CTL), (
    f'{STAGE1_CTL} missing. If Round 3 stage 1 is really gone, uncomment the cell below and spend the '
    '~2.5 h — and SAY SO when reporting, because the control then has a fresh stage 1 too.')
print('control reuses Round 3 stage 1:', STAGE1_CTL)

# --- only if the checkpoint above is really gone -------------------------------------------------
# %cd /content/tnc
# !python -u src/vision/train.py --vocab old --strips-dir {STRIPS} --split data/split_v4.json \
#     --every-share 0.15 --out-dir {DRIVE}/r4-ctl-stage1 \
#     --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 \
#     --eval-every 250 --save-every 250

In [ ]:
# ===== H ARM, STAGE 1 — synthetic only, from BASE, --vocab h (~2.5 h on an L4) =====
# ⚠ The SAME flags as Round 3's stage 1 except --vocab h. Anything else moved here is a second variable.
# ⚠ `-u` is not cosmetic: without it Colab buffers the log and a healthy run looks frozen.
%cd /content/tnc
!python -u src/vision/train.py --vocab h --strips-dir {STRIPS} --split data/split_v4.json \
    --every-share 0.15 --out-dir {STAGE1_H} \
    --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250  # ~nproc-2; T4 (2 vCPU) use 2

In [ ]:
# ===== CONTROL, STAGE 2 — from Round 3's stage 1, --vocab old (~1.7 h + selector) =====
%cd /content/tnc
!python -u src/vision/train.py --vocab old --model {STAGE1_CTL} \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} {SELECT} --select-max-length {SELECT_MAX} \
    --every-share 0.15 --out-dir {DRIVE}/r4-ctl-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 4000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250

In [ ]:
# ===== H ARM, STAGE 2 — from ITS OWN stage 1, --vocab h (~1.7 h + selector) =====
# ⚠ Identical to the control's cell except --vocab, --model and --out-dir.
%cd /content/tnc
import os
assert os.path.isdir(f'{STAGE1_H}/best'), f'{STAGE1_H}/best missing — run the H stage 1 first'
!python -u src/vision/train.py --vocab h --model {STAGE1_H}/best \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} {SELECT} --select-max-length {SELECT_MAX} \
    --every-share 0.15 --out-dir {DRIVE}/r4-h-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 4000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250

In [ ]:
# RESUME after a disconnect: re-run the setup cells, then COPY the cell of the stage you were running
# here and add --resume. It reloads model + optimizer + scheduler from <out-dir>/last and ignores
# --model. ⚠ Keep EVERY other flag identical, --vocab above all.

In [ ]:
# ===== THE PAIRED READ ON REAL-VAL — on the GPU, to keep the Mac cool =====
# Each arm's pick is its `best-edits` checkpoint. ⛔ --score-vocab old is REQUIRED (see the header).
# NEGATIVE mean difference = the H arm needs FEWER corrections. A CI spanning 0 is a NULL.
# ⚠ AFTER A DISCONNECT the machine's disk and every variable are gone (Drive is not): run the setup
# cells KNOBS -> GPU -> Mount -> Copy -> Dependencies first, and NONE of the training cells.
import os
assert 'DRIVE' in globals() and os.path.isdir('/content/tnc/scripts/rung3'), (
    'the runtime was reset: run the setup cells (KNOBS, GPU, Mount Drive, Copy the package, '
    'Dependencies) first — NOT the shakeout or any training cell')
for ck in ['r4-ctl-stage2/best-edits', 'r4-h-stage2/best-edits']:
    assert os.path.isdir(f'{DRIVE}/{ck}'), f'{DRIVE}/{ck} missing on Drive — has that stage finished?'
%cd /content/tnc
!mkdir -p {DRIVE}/r4-read
!python scripts/rung3/paired_arm_score.py --score-vocab old \
    --ctl {DRIVE}/r4-ctl-stage2/best-edits --arm {DRIVE}/r4-h-stage2/best-edits \
    --pool data/real/rung3/_realval_v2r --out {DRIVE}/r4-read/paired_realval_v2r.json

# THE LONG-STRIP COLUMN the round watches: errors by length bucket (short <30 / mid 30-49 /
# long >=50, in OLD ids for both arms) and by kind.
!python scripts/rung3/error_taxonomy.py --checkpoint {DRIVE}/r4-ctl-stage2/best-edits \
    --compare {DRIVE}/r4-h-stage2/best-edits --pool data/real/rung3/_realval_v2r \
    --cache-dir /content/taxonomy_cache --out {DRIVE}/r4-read/taxonomy_realval_v2r.json

## After the run

1. **Download** `best-edits`, `best-real` and `last` from both `MyDrive/tnc/r4-ctl-stage2/` and
   `MyDrive/tnc/r4-h-stage2/`, plus `MyDrive/tnc/r4-read/`.
2. **The paired read above is the real-val answer.** Report the mean difference **with its CI**, the
   sign test, and the long-strip bucket beside it. A CI spanning 0 is a null, and a null is reported
   as one.
3. **Then the owner's hand-test pages**, both picks on the same 20 pages, corrections counted per page.
4. **`examv3` once, on the chosen model only**, as the comparable column. ⛔ If the chosen model is
   H, `eval_omr.py` must first count on the old scale — it does not yet. ⛔ The exam is one-shot per
   round: never read it for this A/B.

In [ ]:
# ===== THE DENSE READ — the class this round was actually opened for (2026-09-18) =====
# `_realval_v2r` is only 5.8% dense (15 of 260 strips over the old 59-id gate), so none of the reads
# so far measured the over-budget strips H and the rail rescued. strips_h1's HELD-OUT side is 14.5%
# (64 of 442) and 68 of its strips do not exist in strips_b8 at all — the LIVE model never saw them.
# Fair for all three models: 0 of these 61 pieces sit on b8's train side, 0 are exam pieces, and
# neither arm's `best-edits` was picked here (that selector read _realval_v2r + _tupletval).
# ⚠ Lengths are counted in OLD ids for every model — the same rule --score-vocab old follows.
# Mirrors scripts/rung3/build_denseval.py, inlined because the zip predates that script.
%cd /content/tnc
import json, os, sys
sys.path.insert(0, 'src/vision')
from data import is_real_val_piece, vocabulary
from modeling import MODEL_ID
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID); tok.add_tokens(vocabulary('old'))
def n_ids(lbl):
    ids = tok(lbl).input_ids
    return len(ids) + (0 if ids and ids[-1] == tok.eos_token_id else 1)
sv = set(json.load(open('data/split_v4.json'))['val_pieces'])
rows = [json.loads(l) for l in open(f'{REAL}/manifest.jsonl') if l.strip()]
held = [r for r in rows if is_real_val_piece(r.get('piece', ''), sv, 0.10)]
for r in held:
    r['old_ids'] = n_ids(r['label'])
ts = json.load(open('data/real/rung3/testset.json'))
exam = {(e['symbtr_file'][:-4] if e.get('symbtr_file', '').endswith('.txt') else e.get('symbtr_file', ''))
        for e in ts['pieces']}
assert not ({r['piece'] for r in held} & exam), 'EXAM LEAK — do not score on this pool'
DENSE, HELD = 'data/real/rung3/_denseval_h1_dense', 'data/real/rung3/_denseval_h1'
for out, keep in [(HELD, held), (DENSE, [r for r in held if r['old_ids'] > 49])]:
    os.makedirs(out, exist_ok=True)
    for r in keep:
        dst = f"{out}/{r['image']}"
        if not os.path.exists(dst):
            os.link(f"{REAL}/{r['image']}", dst)
    with open(f'{out}/manifest.jsonl', 'w') as f:
        for r in keep:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(out, len(keep), 'strips |', sum(1 for r in keep if r['old_ids'] > 59), 'over the old 59-id gate')


In [ ]:
# ===== the four dense reads (~20 min on an L4, my estimate — not measured) =====
# ⚠ Read the DENSE pool first: it is the round's own question. The 442-strip pool is the context.
%cd /content/tnc
!mkdir -p {DRIVE}/r4-read
# 1. the vocabulary question, on dense strips: H against the control, both picks
!python scripts/rung3/paired_arm_score.py --score-vocab old \
    --ctl {DRIVE}/r4-ctl-stage2/best-edits --arm {DRIVE}/r4-h-stage2/best-edits \
    --pool data/real/rung3/_denseval_h1_dense --out {DRIVE}/r4-read/dense_h_vs_ctl.json
# 2. the round's data question, on dense strips: Round 4's control against the LIVE model.
#    ⚠ `last`, not `best-edits` — the unbiased copy, the same rule the 2026-09-17 check settled on.
!python scripts/rung3/paired_arm_score.py --score-vocab old \
    --ctl {DRIVE}/r3-r3a-stage2/best-real --arm {DRIVE}/r4-ctl-stage2/last \
    --pool data/real/rung3/_denseval_h1_dense --out {DRIVE}/r4-read/dense_live_vs_ctl.json
# 3-4. the same two over all 442 held-out strips, for context
!python scripts/rung3/paired_arm_score.py --score-vocab old \
    --ctl {DRIVE}/r4-ctl-stage2/best-edits --arm {DRIVE}/r4-h-stage2/best-edits \
    --pool data/real/rung3/_denseval_h1 --out {DRIVE}/r4-read/held_h_vs_ctl.json
!python scripts/rung3/paired_arm_score.py --score-vocab old \
    --ctl {DRIVE}/r3-r3a-stage2/best-real --arm {DRIVE}/r4-ctl-stage2/last \
    --pool data/real/rung3/_denseval_h1 --out {DRIVE}/r4-read/held_live_vs_ctl.json
